In [1]:
# ============================================================
# 03_pyspark_pipeline.ipynb
# PySpark ML Pipeline - NYC Taxi Dataset
# ============================================================

# 1. Imports

import time
import requests
import pandas as pd

from pydantic import ByteSize
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator


In [2]:
# 2. Start Spark session

spark = (
    SparkSession.builder
    .appName("BDCC PySpark Taxi Pipeline")
    .getOrCreate()
)

spark


In [3]:
# 3. Configuration

RUN_MODE = "local"

if RUN_MODE == "local":
    DATA_PATH = "C:/Users/Tubias/Big Data/Assignment 2/yellow_tripdata_2026-01.parquet"
else:
    DATA_PATH = "gs://your-bucket/taxi/yellow_tripdata_2026-01.parquet"

LOOKUP_PATH = "C:/Users/Tubias/Big Data/Assignment 2/data/raw/taxi/taxi_zone_lookup.csv"

SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

TARGET = "fare_amount"

FEATURES = [
    "trip_distance",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
    "payment_type",
]

FEATURES_COL = "features"
LABEL_COL = "label"

SEED = 42

TEST_SIZE = 0.2
TRAIN_RATIO = 0.8

LOGISTIC_MAX_ITER = 100
LOGISTIC_REGULARIZATION = 0.0

XGB_N_ESTIMATORS = 100
XGB_MAX_DEPTH = 6
XGB_LEARNING_RATE = 0.1

N_JOBS = -1


In [4]:
from collections.abc import Generator
from contextlib import contextmanager
from datetime import timedelta
from pathlib import Path
from time import perf_counter
from typing import TypedDict

class Duration(TypedDict):
    duration: timedelta


def _mem_str(path: str) -> str:
    path = Path(path)
    size = path.stat().st_size
    return ByteSize(size).human_readable(decimal=True)


def download_if_missing(path: str, url: str) -> None:
    path = Path(path)
    if path.exists():
        return

    print(f"Downloading {path.name}...")
    response = requests.get(url)
    response.raise_for_status()
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded {path}, {_mem_str(path)}")


@contextmanager
def timeit(name: str) -> Generator[dict]:
    result = {"duration": timedelta()}
    start_time = perf_counter()
    yield result
    end_time = perf_counter()
    duration = timedelta(seconds=end_time - start_time)
    print(f"{name} took {duration}")
    result["duration"] = duration


In [5]:
# 4. Load data

download_if_missing(DATA_PATH, SOURCE_URL)
download_if_missing(LOOKUP_PATH, LOOKUP_URL)

with timeit("Loading data"):
    df = spark.read.parquet(DATA_PATH)

location_lookup = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(LOOKUP_PATH)
)

df.show(5)


Loading data took 0:00:01.544528
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|   

In [6]:
# 5. Basic dataset inspection

print("Columns:")
print(df.columns)

print("\nSchema:")
df.printSchema()

print("\nNumber of partitions:")
print(df.rdd.getNumPartitions())


Columns:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']

Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = t

In [7]:
# 6. Preprocessing

df = df.select(*(FEATURES + [TARGET]))

df = df.dropna()

df = df.filter(
    (F.col("fare_amount") > 0) &
    (F.col("trip_distance") > 0) &
    (F.col("passenger_count") > 0)
)

df.show(5)


+-------------+---------------+------------+------------+------------+-----------+
|trip_distance|passenger_count|PULocationID|DOLocationID|payment_type|fare_amount|
+-------------+---------------+------------+------------+------------+-----------+
|         0.97|              1|         239|         238|           1|        7.2|
|         5.58|              4|         142|         209|           1|       38.7|
|         2.33|              2|         144|         137|           1|       14.2|
|          1.3|              1|         142|          50|           2|       11.4|
|         5.34|              1|         161|          45|           1|       37.3|
+-------------+---------------+------------+------------+------------+-----------+
only showing top 5 rows


In [ ]:
def benchmark_pyspark(data_path: str, lookup_path: str, scenario="Standard"):
    """
    Runs the exact operation-level benchmark suite detailed in the Databricks experiment.
    Uses PySpark SQL functions and your local `timeit` context manager.
    Scenarios: 'Standard', 'Filtered', 'Filtered_Cached'
    """
    from pyspark.sql import functions as F
    metrics = {}

    # ====================================================
    # 1. READ FILE / INITIAL SETUP
    # ====================================================
    with timeit("read file") as duration:
        if scenario == "Standard":
            df = spark.read.parquet(data_path)
            df_lookup = spark.read.option("header", True).option("inferSchema", True).csv(lookup_path)
            
        elif scenario == "Filtered":
            df = spark.read.parquet(data_path)
            # Filter rows where tip is between $1 and $5
            df = df.filter((F.col("tip_amount") >= 1) & (F.col("tip_amount") <= 5))
            df_lookup = spark.read.option("header", True).option("inferSchema", True).csv(lookup_path)
            
        elif scenario == "Filtered_Cached":
            df = spark.read.parquet(data_path)
            df = df.filter((F.col("tip_amount") >= 1) & (F.col("tip_amount") <= 5))
            # Cache the filtered dataset in memory and force an action to materialize it
            df = df.cache()
            df.count() 
            
            # Broadcast Join Optimization: Tell Spark to keep the small lookup table entirely in RAM
            df_lookup = spark.read.option("header", True).option("inferSchema", True).csv(lookup_path)
            df_lookup = F.broadcast(df_lookup)
            
    metrics["read file"] = duration["duration"]

    # Pre-select columns to simulate generic Databricks evaluation properties
    sa = F.col("trip_distance")
    sb = F.col("fare_amount")
    
    # Safely handle potential division by zero for raw 'Standard' math 
    sb_safe = F.when(sb > 0, sb).otherwise(1.0)

    # ====================================================
    # 2. STANDARD BENCHMARKS
    # ====================================================
    
    with timeit("count") as duration:
        _ = df.count()
    metrics["count"] = duration["duration"]
    
    with timeit("count index") as duration:
        # Spark dataframes don't have a pandas-style explicit index, 
        # but Databricks measures a simple count here to evaluate metadata/plan optimization
        _ = df.count()
    metrics["count index"] = duration["duration"]
    
    with timeit("mean") as duration:
        _ = df.select(F.mean(sa)).collect()
    metrics["mean"] = duration["duration"]
    
    with timeit("standard deviation") as duration:
        _ = df.select(F.stddev(sa)).collect()
    metrics["standard deviation"] = duration["duration"]
    
    with timeit("value counts") as duration:
        _ = df.groupBy(sa).count().collect()
    metrics["value counts"] = duration["duration"]

    # ====================================================
    # 3. SERIES MATH & ARITHMETIC
    # ====================================================
    
    with timeit("series addition") as duration:
        # In PySpark, we use select/withColumn to add transformation logic lazily
        res_add = df.select((sa + sb).alias("res"))
    metrics["series addition"] = duration["duration"]
    
    with timeit("mean of series addition") as duration:
        _ = res_add.select(F.mean("res")).collect()
    metrics["mean of series addition"] = duration["duration"]
    
    with timeit("series multiplication") as duration:
        res_mul = df.select((sa * sb).alias("res"))
    metrics["series multiplication"] = duration["duration"]
    
    with timeit("mean of series multiplication") as duration:
        _ = res_mul.select(F.mean("res")).collect()
    metrics["mean of series multiplication"] = duration["duration"]

    with timeit("complex arithmetic") as duration:
        # Replicating the sin/cos/atan2 trigonometric logic in PySpark expressions
        res_complex = df.select(
            (F.sin(sa) * F.cos(sb_safe) + F.atan2(sa, sb_safe)).alias("res")
        )
    metrics["complex arithmetic"] = duration["duration"]
    
    with timeit("mean of complex arithmetic") as duration:
        _ = res_complex.select(F.mean("res")).collect()
    metrics["mean of complex arithmetic"] = duration["duration"]

    # ====================================================
    # 4. AGGREGATIONS & JOINS
    # ====================================================
    
    with timeit("groupby statistics") as duration:
        _ = (
            df.groupBy("passenger_count")
            .agg(F.mean(sa).alias("mean"), F.stddev(sa).alias("std"))
            .collect()
        )
    metrics["groupby statistics"] = duration["duration"]
    
    # Cast outside the timer to keep the core join execution metric completely pure
    df_lookup = df_lookup.withColumn("LocationID", F.col("LocationID").cast(df.schema["PULocationID"].dataType))
    
    with timeit("join") as duration:
        # Measures lazy query graph execution layout creation
        joined_df = df.join(df_lookup, df["PULocationID"] == df_lookup["LocationID"], how="inner")
    metrics["join"] = duration["duration"]
    
    with timeit("join count") as duration:
        # Forces execution and shuffles/broadcasts data across threads
        _ = joined_df.count()
    metrics["join count"] = duration["duration"]
    
    return metrics

In [ ]:
benchmarks_standard = benchmark_pyspark(DATA_PATH, LOOKUP_PATH, "Standard")


read file took 0:00:00.202196
count took 0:00:00.105038
count index took 0:00:00.075330
mean took 0:00:00.178908
standard deviation took 0:00:00.109254
value counts took 0:00:00.385932
series addition took 0:00:00.006903
mean of series addition took 0:00:00.187036
series multiplication took 0:00:00.008833
mean of series multiplication took 0:00:00.147132
complex arithmetic took 0:00:00.013327
mean of complex arithmetic took 0:00:00.144445
groupby statistics took 0:00:00.247605
join took 0:00:00.005559
join count took 0:00:00.160310


In [ ]:
benchmarks_filtered = benchmark_pyspark(DATA_PATH, LOOKUP_PATH, "Filtered")


read file took 0:00:00.145580
count took 0:00:00.141892
count index took 0:00:00.187959
mean took 0:00:00.170896
standard deviation took 0:00:00.230561
value counts took 0:00:00.365516
series addition took 0:00:00.006181
mean of series addition took 0:00:00.159212
series multiplication took 0:00:00.026791
mean of series multiplication took 0:00:00.172629
complex arithmetic took 0:00:00.008588
mean of complex arithmetic took 0:00:00.170796
groupby statistics took 0:00:00.230794
join took 0:00:00.005745
join count took 0:00:00.149247


In [ ]:
benchmarks_cached = benchmark_pyspark(DATA_PATH, LOOKUP_PATH, "Filtered_Cached")


read file took 0:00:01.831304
count took 0:00:00.043483
count index took 0:00:00.080450
mean took 0:00:00.123106
standard deviation took 0:00:00.075378
value counts took 0:00:00.299817
series addition took 0:00:00.005710
mean of series addition took 0:00:00.090309
series multiplication took 0:00:00.005615
mean of series multiplication took 0:00:00.072714
complex arithmetic took 0:00:00.011379
mean of complex arithmetic took 0:00:00.108183
groupby statistics took 0:00:00.174897
join took 0:00:00.004338
join count took 0:00:00.123385


In [10]:
# 7. Feature vector preparation

assembler = VectorAssembler(
    inputCols=FEATURES,
    outputCol=FEATURES_COL,
    handleInvalid="skip"
)

ml_df = assembler.transform(df).select(
    F.col(FEATURES_COL),
    F.col(TARGET).alias(LABEL_COL)
)

train_df, test_df = ml_df.randomSplit([TRAIN_RATIO, TEST_SIZE], seed=42)

train_df.cache()
test_df.cache()

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())


Train rows: 2040866
Test rows: 510985


In [11]:
# 8. Regression pipeline - GBTRegressor
# Note: PySpark MLlib does not use XGBRegressor by default.
# GBTRegressor is the closest native Spark tree boosting alternative.

reg_model = GBTRegressor(

    featuresCol=FEATURES_COL,
    labelCol=LABEL_COL,
    maxIter=XGB_N_ESTIMATORS,
    maxDepth=XGB_MAX_DEPTH,
    stepSize=XGB_LEARNING_RATE,
    seed=SEED

)

with timeit("Regression training") as duration:
    fitted_reg_model = reg_model.fit(train_df)

regression_train_time = duration["duration"]

reg_predictions = fitted_reg_model.transform(test_df)
reg_predictions.select("prediction", LABEL_COL).show(5)


Regression training took 0:02:22.759834
+-----------------+-----+
|       prediction|label|
+-----------------+-----+
| 36.6686304000603| 25.0|
|8.358901043559602|  3.0|
| 36.6686304000603|  3.0|
| 36.6686304000603| 30.0|
|7.934824032752879|  3.0|
+-----------------+-----+
only showing top 5 rows


In [12]:
# 9. Regression metrics

mae_evaluator = RegressionEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="mae"
)

mse_evaluator = RegressionEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="mse"
)

rmse_evaluator = RegressionEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="rmse"
)

r2_evaluator = RegressionEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="r2"
)

regression_results = {
    "library": "PySpark",
    "task": "regression",
    "model": "GBTRegressor",
    "train_time": regression_train_time,
    "mae": mae_evaluator.evaluate(reg_predictions),
    "mse": mse_evaluator.evaluate(reg_predictions),
    "rmse": rmse_evaluator.evaluate(reg_predictions),
    "r2": r2_evaluator.evaluate(reg_predictions),
}

regression_results


{'library': 'PySpark',
 'task': 'regression',
 'model': 'GBTRegressor',
 'train_time': datetime.timedelta(seconds=142, microseconds=759834),
 'mae': 3.0195412072191994,
 'mse': 51.70294493845573,
 'rmse': 7.190475988309517,
 'r2': 0.8466337419096146}

In [13]:
# 10. Classification pipeline - LogisticRegression

classified_df = df.withColumn(
    "fare_class",
    F.when(F.col(TARGET) < 15, F.lit(0))
     .when(F.col(TARGET) < 40, F.lit(1))
     .otherwise(F.lit(2))
     .cast(IntegerType())
) 

classification_ml_df = assembler.transform(classified_df).select(
    F.col(FEATURES_COL),
    F.col("fare_class").alias(LABEL_COL)
)

TRAIN_RATIO = 0.8 

train_cls_df = (
    classification_ml_df
    .sampleBy(
        LABEL_COL,
        fractions={0: TRAIN_RATIO, 1: TRAIN_RATIO, 2: TRAIN_RATIO},
        seed=42
    )
)

test_cls_df = classification_ml_df.subtract(train_cls_df)

train_cls_df.cache()
test_cls_df.cache()

cls_model = LogisticRegression(

    featuresCol=FEATURES_COL,
    labelCol=LABEL_COL,
    maxIter=LOGISTIC_MAX_ITER,
    regParam=LOGISTIC_REGULARIZATION,
    elasticNetParam=0.0,
    fitIntercept=True,
    family="multinomial"

)

with timeit("Classification training") as duration:
    fitted_cls_model = cls_model.fit(train_cls_df)

classification_train_time = duration["duration"]

cls_predictions = fitted_cls_model.transform(test_cls_df)
cls_predictions.select("prediction", LABEL_COL).show(5)


Classification training took 0:00:07.880971
+----------+-----+
|prediction|label|
+----------+-----+
|       0.0|    2|
|       1.0|    0|
|       0.0|    0|
|       0.0|    0|
|       0.0|    0|
+----------+-----+
only showing top 5 rows


In [14]:
# 11. Classification metrics

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="accuracy"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="f1"
)

classification_results = {
    "library": "PySpark",
    "task": "classification",
    "model": "LogisticRegression",
    "train_time": classification_train_time,
    "accuracy": accuracy_evaluator.evaluate(cls_predictions),
    "precision_weighted": precision_evaluator.evaluate(cls_predictions),
    "recall_weighted": recall_evaluator.evaluate(cls_predictions),
    "f1_weighted": f1_evaluator.evaluate(cls_predictions),
}

classification_results


{'library': 'PySpark',
 'task': 'classification',
 'model': 'LogisticRegression',
 'train_time': datetime.timedelta(seconds=7, microseconds=880971),
 'accuracy': 0.30945327560403285,
 'precision_weighted': 0.3900478877676035,
 'recall_weighted': 0.30945327560403285,
 'f1_weighted': 0.1627197944353662}

In [15]:
# 12. Save results

from pathlib import Path

results_dir = Path("C:/Users/Tubias/Big Data/Assignment 2/results")
results_dir.mkdir(exist_ok=True)

benchmark_results_df = pd.DataFrame([
    {"operation": operation, "library": "PySpark", "duration": duration}
    for operation, duration in benchmarks.items()
])

ml_results_df = pd.DataFrame([
    regression_results,
    classification_results
])

benchmark_results_df.to_csv("C:/Users/Tubias/Big Data/Assignment 2/results/pyspark_benchmark_results.csv", index=False)
ml_results_df.to_csv("C:/Users/Tubias/Big Data/Assignment 2/results/pyspark_pipeline_results.csv", index=False)

benchmark_results_df, ml_results_df


(               operation  library               duration
 0           1. Read Data  PySpark 0 days 00:00:00.946149
 1     2. Count Operation  PySpark 0 days 00:00:00.167010
 2  3. Complex Arithmetic  PySpark 0 days 00:00:00.163343
 3  4. Standard Deviation  PySpark 0 days 00:00:00.326084
 4        5. GroupBy Mean  PySpark 0 days 00:00:00.603726
 5        6. Join & Count  PySpark 0 days 00:00:00.414804,
    library            task               model             train_time  \
 0  PySpark      regression        GBTRegressor 0 days 00:02:22.759834   
 1  PySpark  classification  LogisticRegression 0 days 00:00:07.880971   
 
         mae        mse      rmse        r2  accuracy  precision_weighted  \
 0  3.019541  51.702945  7.190476  0.846634       NaN                 NaN   
 1       NaN        NaN       NaN       NaN  0.309453            0.390048   
 
    recall_weighted  f1_weighted  
 0              NaN          NaN  
 1         0.309453      0.16272  )